# Desafio Inicial SYNAPTA 2026 — Análise de Preços de Aluguel em São Paulo
**Candidato:** Henrique Leandro Ramos Silva  
**Curso:** Bacharelado em Ciência da Computação — UNINASSAU Maceió  
**Dataset:** [Sao Paulo Housing Prices (Kaggle)](https://www.kaggle.com/datasets/renatosn/sao-paulo-housing-prices)

---

## 1. Carregamento e Auditoria Inicial dos Dados
Verificação da dimensionalidade (linhas e colunas), tipos de dados, valores nulos, registros duplicados e anomalias físicas.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
df_raw = pd.read_csv('data.csv')
print(f'Linhas: {df_raw.shape[0]} | Colunas: {df_raw.shape[1]}')
print('\nValores ausentes por coluna:')
print(df_raw.isnull().sum())
print('\nDuplicatas exatas:', df_raw.duplicated().sum())
df_raw.head()

## 2. Identificação de Inconsistências e Preparação dos Dados
- **7 registros com área inferior a 10 m²:** inclui 2 imóveis com `area == 0` m² e 5 imóveis entre 1 m² e 5 m² (ex.: uma casa em condomínio com 1 m², 4 quartos e 3 vagas no Jardim dos Estados).
- **37 registros com `bedrooms == 0`:** sendo 26 do tipo *Studio e kitnet* (onde o conceito aberto sem parede divisória justifica o valor zero) e **11 imóveis do tipo Apartamento (7) ou Casa (4)** com zero quartos, o que configura erro de preenchimento.
- **Decisão de Limpeza:** remoção dos 7 imóveis com `area < 10` m² e dos 11 imóveis de Apartamento/Casa com `bedrooms == 0`, resultando em **11.639 registros consistentes** (99,85% da base original).

In [2]:
mask_valida = (df_raw['area'] >= 10) & ~((df_raw['bedrooms'] == 0) & (df_raw['type'] != 'Studio e kitnet'))
df = df_raw[mask_valida].copy()

# Criação de variáveis auxiliares
df['encargos'] = df['total'] - df['rent']
df['peso_encargos_pct'] = (df['encargos'] / df['rent']) * 100
df['preco_m2'] = df['rent'] / df['area']

print(f'Base original: {len(df_raw)} | Removidos: {len(df_raw) - len(df)} | Base limpa: {len(df)}')
df[['area', 'bedrooms', 'garage', 'rent', 'total', 'encargos', 'preco_m2']].describe().T.round(2)

## 3. Preço Médio vs. Preço Mediano e Ranking de Bairros (n >= 30)
Para evitar que bairros com apenas 1 ou 2 imóveis de altíssimo padrão distorçam o ranking, aplicamos o filtro de pelo menos **30 imóveis por bairro**.

In [3]:
bairros = df.groupby('district').agg(
    qtd=('rent', 'count'),
    media_aluguel=('rent', 'mean'),
    mediana_aluguel=('rent', 'median'),
    mediana_m2=('preco_m2', 'median'),
    mediana_area=('area', 'median')
).query('qtd >= 30')

print('--- 10 BAIRROS MAIS CAROS (n >= 30) ---')
display(bairros.sort_values('media_aluguel', ascending=False).head(10).round(2))

print('--- 10 BAIRROS MAIS BARATOS (n >= 30) ---')
display(bairros.sort_values('media_aluguel', ascending=True).head(10).round(2))

## 4. Comparação com Características do Imóvel (Área, Quartos, Vagas e Tipo)

In [4]:
print('Correlação de Pearson com o Aluguel (rent):')
display(df[['area', 'bedrooms', 'garage', 'rent', 'total', 'preco_m2']].corr().round(3))

print('\nResumo por Tipo de Imóvel:')
display(df.groupby('type').agg(
    qtd=('rent', 'count'),
    media_aluguel=('rent', 'mean'),
    mediana_aluguel=('rent', 'median'),
    mediana_total=('total', 'median'),
    mediana_encargos=('encargos', 'median'),
    mediana_m2=('preco_m2', 'median')
).round(2))